In [2]:
import networkx as nx
import string
import matplotlib.pyplot as plt
import scipy
import random

In [4]:
numbers = [None, None, None]
new_numbers = [1] + [numbers[1]] + [2]
print(new_numbers)

[1, None, 2]


In [80]:
import random
test = {}
test["a"] = 0
test['b'] = 1
test['c'] = 1
sum(test.values())

def removeCards(test):
    num_cards = sum(test.values())
    if num_cards == 0:
        return "No Cards to take"
    random_idx = random.randint(0, num_cards-1)
    for key in test.keys():
        if test[key] > 0:
            if (random_idx - test[key]) < 0:
                test[key]-=1
                return key
            else:
                random_idx-=test[key]
            
    return "You shouldnt be here"
return_values = {"a":0, "b":0, "c": 0}
for i in range(10000):
    test = {"a":0, "b":0, "c": 1}
    return_values[removeCards(test)]+=1
print(return_values)


{'a': 0, 'b': 0, 'c': 10000}


2


1

In [3]:
G = nx.Graph()
## Now make a node map
class Node:
    # Whether or not this spot has a settlement or city on it. 
    def __init__(self, resource, number):
        self.resources = []
        self.numbers = []
        self.resources.append(resource)
        self.numbers.append(number)
        self.usage = None
    def show(self):
        display = f"The resources are: {self.resources} and the numbers are: {self.numbers}"
        return display
class Tile:
    resource = None
    number = -1
    nodes = None
    def __init__(self, resource, number, G):
        self.resource = resource
        self.number = number
        self.nodes = [Node(resource, number) for i in range(6)]
        for i in range(len(self.nodes)-1):
            G.add_edge(self.nodes[i], self.nodes[i+1])
        G.add_edge(self.nodes[5], self.nodes[0])
    def show(self):
        output = ""
        for i, node in enumerate(self.nodes):
            output+= f"Node {str(i)} : {node.show()}\n\n"
        return output 

In [4]:
# When Tile A is to the left of Tile B 

def join0(a, b, G, Tiles):
    neighbors1 = list(G.neighbors(b.nodes[2]))
    neighbors2 = list(G.neighbors(b.nodes[3]))

    G.remove_node(b.nodes[2])
    G.remove_node(b.nodes[3])
    
    tmp1 = b.nodes[2]
    tmp2 = b.nodes[3]

    b.nodes[2] = a.nodes[0]
    for tile in Tiles:
        for i in range(len(tile.nodes)):
            if tile.nodes[i] is tmp1:
                tile.nodes[i] = a.nodes[0]
    for neighbor in neighbors1:
        G.add_edge(b.nodes[2], neighbor)
    
    b.nodes[3] = a.nodes[5]
    for tile in Tiles:
        for i in range(len(tile.nodes)):
            if tile.nodes[i] is tmp2:
                tile.nodes[i] = a.nodes[5]
    for neighbor in neighbors2:
        G.add_edge(b.nodes[3], neighbor)
    
    b.nodes[2].resources.extend(tmp1.resources.copy())
    b.nodes[2].numbers.extend(tmp1.numbers.copy())

    b.nodes[3].resources.extend(tmp2.resources.copy())
    b.nodes[3].numbers.extend(tmp2.numbers.copy())

    G.add_edge(a.nodes[0], b.nodes[1])
    G.add_edge(a.nodes[-1], b.nodes[4])
    return

def join60(a, b, G, Tiles):

    neighbors1 = list(G.neighbors(b.nodes[3]))
    neighbors2 = list(G.neighbors(b.nodes[4]))
    
    G.remove_node(b.nodes[3])
    G.remove_node(b.nodes[4])
    
    tmp1 = b.nodes[3]
    tmp2 = b.nodes[4]
    
    b.nodes[3] = a.nodes[1]
    for tile in Tiles:
        for i in range(len(tile.nodes)):
            if tile.nodes[i] is tmp1:
                tile.nodes[i] = a.nodes[1]
    for neighbor in neighbors1:
        G.add_edge(b.nodes[3], neighbor)
    
    b.nodes[4] = a.nodes[0]
    for tile in Tiles:
        for i in range(len(tile.nodes)):
            if tile.nodes[i] is tmp2:
                tile.nodes[i] = a.nodes[1]
    for neighbor in neighbors2:
        G.add_edge(b.nodes[4], neighbor)
    
    a.nodes[1].resources.extend(tmp1.resources.copy())
    a.nodes[1].numbers.extend(tmp1.numbers.copy())

    a.nodes[0].resources.extend(tmp2.resources.copy())
    a.nodes[0].numbers.extend(tmp2.numbers.copy())

    G.add_edge(a.nodes[1], b.nodes[2])
    G.add_edge(a.nodes[0], b.nodes[4])
    return

def join120(a, b, G, Tiles): #e a
    
    neighbors1 = list(G.neighbors(b.nodes[4]))
    neighbors2 = list(G.neighbors(b.nodes[5]))
    
    G.remove_node(b.nodes[4])
    G.remove_node(b.nodes[5])
    
    tmp1 = b.nodes[4]
    tmp2 = b.nodes[5]

    b.nodes[4] = a.nodes[2]
    for tile in Tiles:
        for i in range(len(tile.nodes)):
            if tile.nodes[i] is tmp1:
                tile.nodes[i] = a.nodes[2]

    for neighbor in neighbors1:
        G.add_edge(b.nodes[4], neighbor)
    
    b.nodes[5] = a.nodes[1]
    for tile in Tiles:
        for i in range(len(tile.nodes)):
            if tile.nodes[i] is tmp2:
                tile.nodes[i] = a.nodes[1]

    for neighbor in neighbors2:
        G.add_edge(b.nodes[5], neighbor)

    
    b.nodes[4].resources.extend(tmp1.resources.copy())
    b.nodes[4].numbers.extend(tmp1.numbers.copy())

    b.nodes[5].resources.extend(tmp2.resources.copy())
    b.nodes[5].numbers.extend(tmp2.numbers.copy())

    G.add_edge(a.nodes[2], b.nodes[4])
    G.add_edge(a.nodes[1], b.nodes[5])
    return
    
# Debugger Function
def displayTile(tile: Node):
    memoryaddys = "Nodes of tile" + str(tile) + "\n"
    for i, node in enumerate(tile.nodes):
        memoryaddys+=(str(i) + ":" + str(node) + "\n")
    return memoryaddys


In [5]:
# Key are the angles which are [0, 60, 120, 180, 240, 300]
# values are the tile objects
tilemaplist = [{0:1, 240: 3, 300: 4},
           {0: 2, 180: 0, 240: 4, 300: 5},
           {180: 1, 240: 5, 300: 6},
           {0:4, 60: 0, 240: 7, 300: 8},
           {0:5, 60:1, 120:0, 180:3, 240: 8, 300: 9},
           {0:6, 60: 2, 120: 1, 180: 4, 240: 9, 300: 10},
           {120:2, 180:5, 240:10, 300:11},
           {0:8, 60:3,300:12},
           {0:9, 60:4, 120:3, 180:7, 240:12,300:13},
           {0:10, 60:5, 120:4, 180:8, 240:13, 300:14},
           {0:11, 60:6, 120:5, 180:9, 240:14, 300:15},
           {120:6, 180: 10, 240:15},
           {0:13, 60:8,120:7,300:16},
           {0:14, 60:9, 120:8,180:12,240:16, 300:17},
           {0:15, 60:10, 120:9, 180:13, 240:17, 300:18},
           {60:11, 120:10, 180:14, 240:18},
           {0:17, 60:13, 120:12},
           {0:18, 60:14, 120:13, 180:16},
           {60:15, 120:14, 180:17}]

In [10]:
resources = ["rock"] * 3 + ["mud"] * 3 + ["wheat"] * 4 + ["tree"] * 4 + ["sheep"] * 4
random.shuffle(resources)
numbers = [3, 4, 5, 6, 8, 9, 10, 11] * 2 + [2, 12]
random.shuffle(numbers)
interval = random.randint(0, 19)
resources = resources[0:interval] + ['desert'] + resources[interval:]
numbers = numbers[0:interval] + [-1] + numbers[interval:]

In [11]:
Board = nx.Graph()
tiles = []
for i in range(19):
    tiles.append(Tile(resource=resources.pop(), number = numbers.pop(), G = Board))


In [12]:
for i, tilemap in enumerate(tilemaplist):
    print(i)
    for direction in list(tilemap.keys()):
        if direction == 0:
            join0(tiles[i], tiles[tilemap[direction]], Board, tiles)
        elif direction == 60:
            join60(tiles[i], tiles[tilemap[direction]], Board, tiles)
        elif direction == 120:
            join120(tiles[i], tiles[tilemap[direction]], Board, tiles)
        elif direction == 180:
            join0(tiles[tilemap[direction]], tiles[i], Board, tiles)
        elif direction == 240:
            join60(tiles[tilemap[direction]], tiles[i], Board, tiles)
        else:
            join120(tiles[tilemap[direction]], tiles[i], Board, tiles)



0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18


In [ ]:
#Test integrity of Tile 1
print(tiles[0].nodes[0] is tiles[1].nodes[2])
print(tiles[0].nodes[5] is tiles[1].nodes[3])
print(tiles[0].nodes[5] is tiles[4].nodes[1])

# Tes integrity of Tile 5
print(tiles[5].nodes[0] is tiles[2].nodes[4])
print(tiles[5].nodes[0] is tiles[6].nodes[2])

print(tiles[5].nodes[1] is tiles[2].nodes[3])
print(tiles[5].nodes[1] is tiles[1].nodes[5])

print(tiles[5].nodes[2] is tiles[1].nodes[4])
print(tiles[5].nodes[2] is tiles[4].nodes[0])

print(tiles[5].nodes[3] is tiles[4].nodes[5])
print(tiles[5].nodes[3] is tiles[9].nodes[1])

print(tiles[5].nodes[4] is tiles[9].nodes[0])
print(tiles[5].nodes[4] is tiles[10].nodes[2])

print(tiles[5].nodes[5] is tiles[6].nodes[3])
print(tiles[5].nodes[5] is tiles[10].nodes[1])

print()


True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [124]:
def displayTile(tile: Node):
    memoryaddys = "Nodes of tile" + str(tile) + "\n"
    for i, node in enumerate(tile.nodes):
        memoryaddys+=(str(i) + ":" + str(node) + "\n")
    return memoryaddys
board = nx.Graph()
a = Tile("wheat", 8, board)
b = Tile("rock", 6, board)
c = Tile("sheep", 10, board)
d = Tile("urmom", 9, board)
e = Tile("Water", 6, board)
f = Tile("derby", 12, board)
tilelist = [a,b,c,d,e,f]
join0(a, b, board)
#join60(d, a, board)
test = list(board.neighbors(a.nodes[4]))
join120(e, a, board, tilelist)
#join0(b, c, G)
#join60(e, b, G)
#join120(f, b, G)
print(displayTile(a))
print(displayTile(b))
print(displayTile(d))
print(displayTile(e))

[<__main__.Node object at 0x7121093af6e0>, <__main__.Node object at 0x7121093adf40>]
[<__main__.Node object at 0x7121093ac320>, <__main__.Node object at 0x7121093aeff0>, <__main__.Node object at 0x7121093af650>]
Nodes of tile<__main__.Tile object at 0x7121093af920>
0:<__main__.Node object at 0x7121093aeff0>
1:<__main__.Node object at 0x7121093ac6b0>
2:<__main__.Node object at 0x7121093ae5d0>
3:<__main__.Node object at 0x7121093af6e0>
4:<__main__.Node object at 0x7121093ae600>
5:<__main__.Node object at 0x7121093ae570>

Nodes of tile<__main__.Tile object at 0x7121093afb60>
0:<__main__.Node object at 0x7121093ac440>
1:<__main__.Node object at 0x7121093ae150>
2:<__main__.Node object at 0x7121093aeff0>
3:<__main__.Node object at 0x7121093ae570>
4:<__main__.Node object at 0x7121093af650>
5:<__main__.Node object at 0x7121093afa40>

Nodes of tile<__main__.Tile object at 0x7121093affb0>
0:<__main__.Node object at 0x7121093aeb70>
1:<__main__.Node object at 0x7121093ad160>
2:<__main__.Node objec

In [ ]:
G = nx.Graph()
a = Tile("wheat", 8, G)
b = Tile("rock", 6, G)
c = Tile("sheep", 10, G)
d = Tile("wood", 3, G)
join0(a, b, G)
join60(a,d,G)
join120(a,c, G)

In [ ]:
print(a.show())

Node 0 : The resources are: ['wheat', 'rock', 'wood'] and the numbers are: [8, 6, 3]

Node 1 : The resources are: ['wheat', 'wood', 'sheep'] and the numbers are: [8, 3, 10]

Node 2 : The resources are: ['wheat', 'sheep'] and the numbers are: [8, 10]

Node 3 : The resources are: ['wheat'] and the numbers are: [8]

Node 4 : The resources are: ['wheat'] and the numbers are: [8]

Node 5 : The resources are: ['wheat', 'rock'] and the numbers are: [8, 6]


